# Cropper — full card 63×88 mm

The cropper **does not train**. Training: `notebooks/train_detector.ipynb`.

Output: full portrait card in color. OCR uses `OCR_TEMPLATE` on this image — name/HP strips are not cropped here.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

def find_repo() -> Path:
    here = Path(".").resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "cropper" / "rectify.py").is_file() and (p / "src" / "detection").is_dir():
            return p
    raise FileNotFoundError(
        "Open this notebook from the PTCG Scanner repo (folders src/cropper/ and src/detection/)."
    )


REPO = find_repo()

sys.path.insert(0, str(REPO / "src" / "cropper"))
from crop_from_obb import DEFAULT_IMGSZ, DEFAULT_WEIGHTS, run
from rectify import DEFAULT_DPI, card_size_px, crop_result

WEIGHTS = DEFAULT_WEIGHTS
OUT_DIR = REPO / "src" / "cropper" / "output" / "cards"
CONF = 0.8
IMGSZ = DEFAULT_IMGSZ
DPI = DEFAULT_DPI
REFINE = False
INSET = 0.02
ENHANCE = True
FRAME = False
MAX_SHOW = 8

STEMS = ["IMG_6674", "IMG_6723", "IMG_7117", "IMG_6654", "IMG_6709"]

def find_images(stems: list[str] | None) -> list[Path]:
    dataset = REPO / "data" / "detection"
    if not stems:
        return sorted((dataset / "test" / "images").glob("*.jpg"))
    found = []
    for stem in stems:
        hits = list(dataset.glob(f"*/images/*{stem}*"))
        if not hits:
            print(f"not found {stem}")
            continue
        found.append(hits[0])
    return found

SOURCES = find_images(STEMS)
print(f"Pesos : {WEIGHTS}  exists={WEIGHTS.exists()}")
print(f"Saída : {OUT_DIR}  {card_size_px(DPI)[0]}×{card_size_px(DPI)[1]} px")
print(f"refine={REFINE}  inset={INSET}  enhance={ENHANCE}  frame={FRAME}  fotos={len(SOURCES)}")


## 1. Recortar


In [ ]:
if not WEIGHTS.exists():
    raise FileNotFoundError(f"Train the detector first. Missing: {WEIGHTS}")
if not SOURCES:
    raise FileNotFoundError("Nenhuma foto.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(WEIGHTS))
saved: list[Path] = []
for src in SOURCES:
    saved.extend(
        run(
            source=src,
            weights=WEIGHTS,
            out_dir=OUT_DIR,
            conf=CONF,
            imgsz=IMGSZ,
            dpi=DPI,
            refine=REFINE,
            inset=INSET,
                        enhance=ENHANCE,
            frame=FRAME,
            model=model,
        )
    )
print(f"{len(saved)} recorte(s) em {OUT_DIR}")


## 2. Cropped cards (63x88 mm)


In [ ]:
saved_show = list(saved)[:MAX_SHOW]
if not saved_show:
    print("No cropped cards.")
else:
    cols = min(4, len(saved_show))
    rows = int(np.ceil(len(saved_show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 4.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, path in zip(axes, saved_show):
        bgr = cv2.imread(str(path))
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(path.name[:28], fontsize=8)
        ax.axis("off")
    for ax in axes[len(saved_show):]:
        ax.axis("off")
    fig.suptitle("Full card 63x88 mm")
    plt.tight_layout()
    plt.show()
